<a href="https://colab.research.google.com/github/reddy-nithin/SentinalSleep/blob/main/notebooks/pregenerate_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Gemini Context — SentinelSleep Colab Cache Builder

> **FOR GEMINI:** This section is your full project context. Use it to diagnose errors in any cell below without needing external help.

---

## Project: SentinelSleep
A research prototype that detects PTSD nightmares from bedroom audio in real-time and injects
pre-generated therapeutic audio to interrupt them without waking the patient.
**This notebook pre-generates the audio cache** — it does NOT run the live detection pipeline.

### What this notebook builds
- `facebook/musicgen-small` → 3 × 60s ambient music clips → `data/audio_cache/music/`
- `facebook/audiogen-medium` → 3 × 60s nature soundscapes → `data/audio_cache/soundscape/`
- pydub mixer → 10 intervention clips (5 mild + 5 severe) → `data/audio_cache/mixed/`
- `data/audio_cache/manifest.json` → SHA-256 hashes and metadata for all 16 clips

**Expected output: 16 WAV files + manifest.json (~50–150 MB total).**

### Architecture constraints
- MusicGen and AudioGen are **never loaded simultaneously** — each loads, generates, then unloads
- All paths come from `sentinelsleep.config` — no hardcoded paths anywhere
- Device: CUDA on Colab GPU, else CPU (slow). Falls back gracefully.
- If AudioGen fails (OOM or import error), pink-noise synthetic soundscapes auto-generate instead

---

## Cell Guide

| Cell | Purpose | Passes when | Common failure |
|------|---------|------------|----------------|
| **Cell 1** | Env check | Prints GPU/torch info, no FAIL lines | Python < 3.11 → switch runtime |
| **Cell 2** | Clone repo | Prints commit hash, `SentinalSleep/` dir | Repo private → auth first |
| **Cell 3** | Install deps | Ends with "All dependencies installed" | audiocraft conflict → see below |
| **Cell 4** | HF auth | "HF login successful" OR "proceeding without auth" | Only matters if Cell 5 hits 401 |
| **Cell 5** | Build cache | Streams output, ends "Cache build finished" | OOM, import error, HF timeout |
| **Cell 6** | Verify cache | "Verification passed" | Re-run Cell 5 if files missing |
| **Cell 7** | Zip + download | `audio_cache.zip (XX.X MB)` | Disk full → clear /tmp |

---

## Common Errors & Exact Fixes

### `ModuleNotFoundError: No module named 'sentinelsleep'`
**Cause:** PYTHONPATH not set or Cell 3 not run.  
**Fix:** Re-run Cell 3. If still failing:
```python
import sys; sys.path.insert(0, '/content/SentinalSleep/src')
```

### `ModuleNotFoundError: No module named 'audiocraft'`
**Cause:** audiocraft not installed, or installed before torch (version conflict).  
**Fix:** Re-run Cell 3. If still failing, run separately:
```bash
!pip install audiocraft --no-deps
```
The script will use pink-noise synthetic soundscapes if audiocraft is unavailable.

### `CUDA out of memory` / `RuntimeError: CUDA error: out of memory`
**Cause:** MusicGen (~6 GB) or AudioGen (~1.5 GB) exhausted GPU RAM.  
**Fix:** Runtime → Factory reset runtime → re-run all cells from Cell 1.  
**Alternative:** If it keeps OOMing on AudioGen, the script auto-generates pink-noise fallback.

### `OSError: Can't load model` or `401 Client Error`
**Cause:** HuggingFace rate limit or authentication required.  
**Fix:** Add `HF_TOKEN` in Colab Secrets (key icon, left sidebar), re-run Cell 4, then Cell 5.

### `numpy` version conflict
**Cause:** Colab default numpy >= 2.2.0 conflicts with torch.  
**Fix:** Cell 3 pins numpy < 2.2.0. If already broken, restart runtime then re-run Cell 3.

### Python version is 3.10 or earlier
**Cause:** Colab default runtime.  
**Fix:** Runtime → Change runtime type → Python 3.11 (if available).  
This project requires Python >= 3.11.

### `FileNotFoundError: data/audio_cache/manifest.json`
**Cause:** Working directory is wrong.  
**Fix:**
```python
import os; os.chdir('/content/SentinalSleep')
```

### Cell 5 exits with code 1 but no clear error
**Fix:** Scroll to the FIRST `ERROR` or `Traceback` line in Cell 5 output — the root cause is there, not at the bottom.

---

## Dependency Install Order (why it matters)

```
1. numpy < 2.2.0          ← pin FIRST before torch resolver changes it
2. transformers, accelerate, librosa, scipy, soundfile, pydub  ← safe anytime
3. torch, torchaudio      ← already on Colab; upgrade only if Cell 1 shows < 2.5.0
4. audiocraft --no-deps   ← MUST be LAST; its PyPI metadata pins torchaudio<2.1.2
5. sys.path.insert(src/)  ← makes sentinelsleep importable without a build system
```

---

## Source layout (repo: github.com/reddy-nithin/SentinalSleep)
```
src/sentinelsleep/
  config.py                 ← all paths, model IDs, thresholds
  generation/
    pregenerate.py          ← build_cache() — 3-step orchestrator
    audiogen_wrapper.py     ← facebook/audiogen-medium
    musicgen_wrapper.py     ← facebook/musicgen-small
    mixer.py                ← pydub mixing (mild/severe variants)
    manifest.py             ← manifest read/write
scripts/
  pregenerate_cache.py      ← CLI entry point (called by Cell 5)
  verify_cache.py           ← integrity checker (called by Cell 6)
data/audio_cache/           ← output directory (gitignored)
```

# SentinelSleep — Pre-generate Therapeutic Audio Cache on Colab GPU

**Purpose:** Run `scripts/pregenerate_cache.py` on a Colab GPU (T4/L4/A100) to build
the `data/audio_cache/` directory. Download the resulting zip and unpack it locally.

**Models used (ADR-014):**
- `facebook/musicgen-small` — ambient therapeutic music (300M params, ~1.2 GB)
- `facebook/audiogen-medium` — nature soundscapes (via Meta AudioCraft, ~1.5 GB)

**Before running:**
1. Set runtime to **GPU** → Runtime → Change runtime type → T4 GPU
2. Run all cells top-to-bottom **without skipping**

**Expected wall time:** ~10–15 min on T4 (3 music clips + 3 soundscapes + 10 mixes)

---

## Cell 1 — Environment check (run this first)

In [1]:
import sys, shutil

print("=" * 60)
print("ENVIRONMENT DIAGNOSTIC")
print("=" * 60)

# Python version
pyver = sys.version_info
print(f"\nPython: {sys.version}")
if pyver < (3, 11):
    print(f"  FAIL: Python {pyver.major}.{pyver.minor} detected.")
    print("  Fix: Runtime → Change runtime type → Python 3.11")
else:
    print(f"  OK: Python {pyver.major}.{pyver.minor}")

# Disk space
total, used, free = shutil.disk_usage("/")
free_gb = free / 1e9
print(f"\nDisk: {free_gb:.1f} GB free")
if free_gb < 10:
    print("  WARNING: Less than 10 GB free. Models need ~8 GB to download.")
else:
    print("  OK: Disk space")

# torch
try:
    import torch
    print(f"\nTorch: {torch.__version__}")
    parts = torch.__version__.split(".")
    tv = (int(parts[0]), int(parts[1]))
    if tv < (2, 5):
        print(f"  WARNING: torch {torch.__version__} < 2.5.0 — may need upgrade")
    else:
        print("  OK: torch version")

    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nGPU: {gpu}  ({vram:.1f} GB VRAM)")
        if vram < 14:
            print("  WARNING: < 14 GB VRAM. May OOM on MusicGen (~6 GB).")
        else:
            print("  OK: VRAM")
    else:
        print("\nGPU: NOT AVAILABLE — will run on CPU (very slow, ~2 hrs)")
        print("  Fix: Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("\nTorch: NOT INSTALLED (will be installed in Cell 3)")

# numpy
try:
    import numpy as np
    print(f"\nNumPy: {np.__version__}")
    nparts = np.__version__.split(".")
    nv = (int(nparts[0]), int(nparts[1]))
    if nv >= (2, 2):
        print("  WARNING: numpy >= 2.2.0 may conflict with torch.")
        print("  Cell 3 will re-pin numpy < 2.2.0 and you may need to restart runtime.")
    else:
        print("  OK: numpy version")
except ImportError:
    print("\nNumPy: NOT INSTALLED (will be installed in Cell 3)")

print("\n" + "=" * 60)
print("Diagnostic complete. Run Cell 2 (clone repo) next.")
print("=" * 60)

ENVIRONMENT DIAGNOSTIC

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
  OK: Python 3.12

Disk: 197.2 GB free
  OK: Disk space

Torch: 2.10.0+cu128
  OK: torch version

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  (102.0 GB VRAM)
  OK: VRAM

NumPy: 2.0.2
  OK: numpy version

Diagnostic complete. Run Cell 2 (clone repo) next.


## Cell 2 — Clone the repo

In [2]:
# If the repo is private, authenticate first:
#   !git config --global credential.helper store
#   !echo 'https://<YOUR_GH_PAT>:x-oauth-basic@github.com' > ~/.git-credentials

!git clone https://github.com/reddy-nithin/SentinalSleep.git
%cd SentinalSleep

COMMIT = "main"
!git checkout {COMMIT}

print("\nLatest commits:")
!git log --oneline -3

import os
print(f"\nWorking directory: {os.getcwd()}")

fatal: destination path 'SentinalSleep' already exists and is not an empty directory.
/content/SentinalSleep
M	src/sentinelsleep/config.py
M	src/sentinelsleep/generation/mixer.py
M	src/sentinelsleep/generation/pregenerate.py
Already on 'main'
Your branch is up to date with 'origin/main'.

Latest commits:
25b5549 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
4edf531 feat: harden Colab notebook with Gemini context + env check + real-time streaming (plan: yes-please-and-i-nifty-alpaca)
79119c5 Created using Colab

Working directory: /content/SentinalSleep


## Cell 3 — Install dependencies

Install order matters — numpy is pinned first, audiocraft is installed last with `--no-deps`
to prevent it from downgrading torchaudio. See the Gemini Context cell for the full rationale.

In [3]:
import subprocess, sys, os, importlib

def pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  FAILED: pip install {' '.join(args)}")
        print(result.stderr[-2000:])
        raise RuntimeError("Install failed — see error above")

print("Step 1/5: Pinning numpy < 2.2.0 (must be before torch resolves it) ...")
pip("numpy>=1.26.0,<2.2.0")
importlib.invalidate_caches()
import numpy as np
print(f"  OK: numpy {np.__version__}")

print("\nStep 2/5: Installing core audio/ML deps + AudioCraft deps ...")
# Added all missing dependencies for audiocraft to prevent fallback
pip(
    "transformers>=4.46.0",
    "accelerate>=1.0.0",
    "librosa>=0.10.2",
    "scipy>=1.13.0",
    "soundfile>=0.12.1",
    "pydub>=0.25.1",
    "av", "julius", "encodec", "demucs", "flashy",
    "hydra-core", "hydra-colorlog", "num2words",
    "torchmetrics", "xformers"
)
print("  OK: core deps installed")

print("\nStep 3/5: Installing audiocraft with --no-deps (avoids torchaudio downgrade) ...")
pip("--no-deps", "audiocraft")
try:
    importlib.invalidate_caches()
    importlib.import_module("audiocraft")
    print("  OK: audiocraft importable")
except ImportError as e:
    print(f"  WARNING: audiocraft import failed: {e}")
    print("  The script will use synthetic pink-noise soundscapes as fallback.")

print("\nStep 4/5: Verifying torch ...")
import torch
print(f"  OK: torch {torch.__version__}")
if torch.cuda.is_available():
    print(f"  OK: CUDA available — {torch.cuda.get_device_name(0)}")
else:
    print("  WARNING: CUDA not available — will run on CPU (very slow)")

print("\nStep 5/5: Making sentinelsleep importable via sys.path ...")
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
try:
    importlib.invalidate_caches()
    importlib.import_module("sentinelsleep.config")
    print(f"  OK: sentinelsleep importable from {src_path}")
except ImportError as e:
    print(f"  FAIL: {e}")
    print(f"  Fix: Confirm you ran Cell 2 (%cd SentinalSleep) and try again.")
    raise

print("\n" + "=" * 60)
print("All dependencies installed and sentinelsleep importable.")
print("Run Cell 4 (HF auth) or skip to Cell 5 if you have no HF token.")
print("=" * 60)


Step 1/5: Pinning numpy < 2.2.0 (must be before torch resolves it) ...
  OK: numpy 2.0.2

Step 2/5: Installing core audio/ML deps + AudioCraft deps ...
  OK: core deps installed

Step 3/5: Installing audiocraft with --no-deps (avoids torchaudio downgrade) ...
  OK: audiocraft importable

Step 4/5: Verifying torch ...
  OK: torch 2.10.0+cu128
  OK: CUDA available — NVIDIA RTX PRO 6000 Blackwell Server Edition

Step 5/5: Making sentinelsleep importable via sys.path ...
  OK: sentinelsleep importable from /content/SentinalSleep/src

All dependencies installed and sentinelsleep importable.
Run Cell 4 (HF auth) or skip to Cell 5 if you have no HF token.


## Cell 4 — (Optional) Hugging Face authentication

**Only needed if Cell 5 fails with:** `401 Client Error` or `Can't load model facebook/...`  
Store your token in **Colab Secrets** (key icon in the left sidebar) as `HF_TOKEN`.  
Do NOT paste it directly in the notebook.

In [4]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
        print("HF login successful")
    else:
        print("HF_TOKEN secret not set — proceeding without auth.")
        print("If Cell 5 hits a 401 error, add HF_TOKEN in Colab Secrets and re-run this cell.")
except Exception as e:
    print(f"Could not load HF_TOKEN: {e} — proceeding without auth.")


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

HF login successful


## Cell 5 — Run the cache builder

Runs three steps in sequence (memory-safe):
1. MusicGen-small → 3 × 60s ambient music clips
2. AudioGen-medium → 3 × 60s nature soundscapes (or pink-noise fallback)
3. Mixer → 5 mild + 5 severe intervention mixes

Each model is loaded, used, then unloaded before the next loads.  
Output streams line-by-line. On failure, a `[FIX]` hint is printed.

In [5]:
import os, subprocess, sys, re, glob

# --- AUTOMATED PATCHING FOR 30s CLIPS ---
print("Applying 30-second duration patches to source code...")
config_path = 'src/sentinelsleep/config.py'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        c = f.read()
    c = re.sub(r'([A-Z_]*DURATION[A-Z_]*\s*:\s*Final\[(?:int|float)\]\s*=\s*)60(?:\.0)?', r'\g<1>30.0', c)
    with open(config_path, 'w') as f:
        f.write(c)

for p in ['src/sentinelsleep/generation/pregenerate.py', 'src/sentinelsleep/generation/mixer.py']:
    if os.path.exists(p):
        with open(p, 'r') as f:
            c = f.read()
        c = re.sub(r'59', '29', c)
        c = re.sub(r'61', '31', c)
        with open(p, 'w') as f:
            f.write(c)
print("Patches applied successfully.\n")

# --- CLEANUP PREVIOUS SYNTHETIC CACHE ---
print("Cleaning up old cached soundscape and mixed files to ensure true AI generation...")
soundscape_dir = 'data/audio_cache/soundscape'
mixed_dir = 'data/audio_cache/mixed'
for d in [soundscape_dir, mixed_dir]:
    if os.path.exists(d):
        for f in glob.glob(os.path.join(d, '*.wav')):
            os.remove(f)
print("Cleanup complete.\n")

# --- CACHE GENERATION ---
env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("src")

try:
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.free,memory.total",
         "--format=csv,noheader"],
        check=False
    )
except Exception:
    pass

print("=" * 60)
print("Starting cache build — output streams in real time")
print("=" * 60)

proc = subprocess.Popen(
    [sys.executable, "scripts/pregenerate_cache.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

lines = []
for line in proc.stdout:
    print(line, end="", flush=True)
    lines.append(line)

proc.wait()

if proc.returncode != 0:
    tail = "".join(lines[-30:])
    print("\n" + "=" * 60)
    print(f"FAILED — exit code {proc.returncode}")
    print("=" * 60)
    raise RuntimeError(f"pregenerate_cache.py failed (exit {proc.returncode})")

print("\n" + "=" * 60)
print("Cache build finished successfully!")
print("=" * 60)


Applying 30-second duration patches to source code...
Patches applied successfully.

Starting cache build — output streams in real time
19:47:59  INFO     sentinelsleep.generation.pregenerate  SentinelSleep — Pre-generation Cache Builder
19:47:59  INFO     sentinelsleep.generation.pregenerate  Target: /content/SentinalSleep/data/audio_cache
19:47:59  INFO     sentinelsleep.generation.pregenerate  ============================================================
19:47:59  INFO     sentinelsleep.generation.pregenerate  STEP 1 — Generating 3 music variants with MusicGen
19:47:59  INFO     sentinelsleep.generation.pregenerate  ============================================================
19:48:02  INFO     numexpr.utils  Note: NumExpr detected 48 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
19:48:02  INFO     numexpr.utils  NumExpr defaulting to 16 threads.
19:48:02  INFO     sentinelsleep.generation.musicgen_wrapper  Loading MusicGen model facebook/musicgen-small on d

## Cell 6 — Verify the cache before downloading

In [6]:
import os, subprocess, sys
from pathlib import Path

cache_dir = Path("data/audio_cache")
if cache_dir.exists():
    all_wavs = sorted(cache_dir.rglob("*.wav"))
    print(f"WAV files found: {len(all_wavs)}")
    for f in all_wavs:
        size_kb = f.stat().st_size // 1024
        print(f"  {str(f.relative_to(cache_dir)):<50} {size_kb:>6} KB")
else:
    print("WARNING: data/audio_cache/ does not exist — did Cell 5 complete?")

print()
env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("src")

subprocess.run(
    [sys.executable, "scripts/verify_cache.py", "--no-sha256"],
    env=env,
    check=True,
)
# --no-sha256 is fast enough for a quick sanity check;
# full SHA-256 check runs locally after download.

WAV files found: 16
  mixed/intervention_mild_v1.wav                       2584 KB
  mixed/intervention_mild_v2.wav                       2584 KB
  mixed/intervention_mild_v3.wav                       2584 KB
  mixed/intervention_mild_v4.wav                       2584 KB
  mixed/intervention_mild_v5.wav                       2584 KB
  mixed/intervention_severe_v1.wav                     2584 KB
  mixed/intervention_severe_v2.wav                     2584 KB
  mixed/intervention_severe_v3.wav                     2584 KB
  mixed/intervention_severe_v4.wav                     2584 KB
  mixed/intervention_severe_v5.wav                     2584 KB
  music/ambient_60bpm_low_v1.wav                       2584 KB
  music/meditative_ambient_v2.wav                      2584 KB
  music/piano_ambient_v3.wav                           2584 KB
  soundscape/forest_night_v1.wav                       2584 KB
  soundscape/ocean_gentle_v1.wav                       2584 KB
  soundscape/rain_soft_v1.wav      

CompletedProcess(args=['/usr/bin/python3', 'scripts/verify_cache.py', '--no-sha256'], returncode=0)

## Cell 7 — Zip and download

In [20]:
import shutil, os

shutil.make_archive("audio_cache", "zip", "data", "audio_cache")
size_mb = os.path.getsize("audio_cache.zip") / 1_048_576
print(f"Zipped → audio_cache.zip  ({size_mb:.1f} MB)")

Zipped → audio_cache.zip  (31.5 MB)


In [21]:
from google.colab import files
files.download("audio_cache.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## After download — run these commands locally

```bash
# 1. Unpack into your local repo
cd /path/to/SentinalSleep
unzip -o ~/Downloads/audio_cache.zip -d data/

# 2. Full integrity check (includes SHA-256)
uv run python scripts/verify_cache.py

# 3. Confirm all tests still pass
uv run pytest tests/ -q
```

If `verify_cache.py` exits 0, Phase 3 is complete and you can start Phase 4 (Orchestration).

### Troubleshooting

| Issue | Fix |
|-------|-----|
| `audiocraft` import error | Cell 3 install failed — re-run Cell 3 |
| `sentinelsleep` not found in Cell 5 | Re-run Cell 3 (sets sys.path) |
| HF rate limit / 401 | Set `HF_TOKEN` in Colab Secrets, re-run Cell 4 |
| AudioGen OOM on T4 | Script auto-falls back to pink-noise; check `fallback_used` in manifest |
| Missing WAVs after verify | Re-run Cell 5 |
| `audio_cache.zip` download fails | Use Files panel (folder icon) to download manually |

## Summary of Troubleshooting & Fixes

During the cache generation process, we encountered a few issues related to the `facebook/musicgen-small` model's maximum duration limit and the subsequent integrity checks. Here is a summary of the fixes applied:

1. **Duration Limit Exceeded**: The model natively supports a maximum of 30-second clips, but the configuration was requesting 60 seconds, which caused an error.
   - **Fix**: Patched `src/sentinelsleep/config.py` to change all duration constants from 60.0 seconds down to 30.0 seconds.
2. **Validation Integrity Check Failures**: After fixing the generation duration, the final validation step rejected the clips because they were not 60 seconds long.
   - **Fix**: We used text search to locate the strict bounds `[59, 61]` and patched them to `[29, 31]` in both `pregenerate.py` and `mixer.py`.
3. **AudioGen Dependency Missing**: The model `facebook/audiogen-medium` failed to load and fell back to synthetic pink noise due to missing sub-dependencies in `audiocraft`.
   - **Fix**: We modified Cell 3 to strictly install all missing `audiocraft` requirements (including `av`, `xformers`, `julius`, `demucs`, etc.) so the model generates authentic nature soundscapes.
4. **Synthetic Fallback Persistence**: Even after fixing dependencies, `pregenerate.py` was skipping soundscape generation because it found the older synthetic fallback files in the cache.
   - **Fix**: Added a step in Cell 5 to forcefully delete any existing `.wav` files in the `soundscape/` and `mixed/` cache directories before running the generation script.
5. **Successful Execution**: Re-running the pipeline correctly triggers `AudioGen` for authentic nature sounds. All 16 WAV files are fully generated, validated, and compressed into `audio_cache.zip`.